In [ ]:
import kaggle_benchmarks as kbench
import json
import re
from datetime import datetime

# ----------------------------
# Global trace store
# ----------------------------
TRACE_LOG = []

FAILURE_MODES = [
    "failure_to_recognize_key_aspects",
    "hallucination",
    "misapplication_of_equation_or_model",
    "incorrect_factual_knowledge",
    "calculation_error",
]

CANONICAL_NUMERIC_ANSWER = 0.192
ABS_TOL = 0.002  # robust for 3 s.f. and outputs like 0.193

# ----------------------------
# Helpers
# ----------------------------
def extract_json(text):
    if not text:
        return None

    fence = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fence:
        blob = fence.group(1)
    else:
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end == -1 or end <= start:
            return None
        blob = text[start:end + 1]

    try:
        return json.loads(blob)
    except Exception:
        return None

def safe_get_attr(obj, attr_name, default=None):
    try:
        return getattr(obj, attr_name, default)
    except Exception:
        return default

def build_trace(
    *,
    task_id,
    llm,
    prompt,
    response,
    parsed,
    final_answer,
    normalized_answer,
    passed,
    failure_mode,
):
    return {
        "timestamp_utc": datetime.utcnow().isoformat() + "Z",
        "task_id": task_id,
        "model": str(llm),
        "pass": bool(passed),
        "failure_mode": failure_mode,
        "final_answer": final_answer,
        "normalized_answer": normalized_answer,
        "raw_output": response,
        "parsed_output": parsed,
        "prompt": prompt,
        "tokens_input": safe_get_attr(llm, "last_input_tokens"),
        "tokens_output": safe_get_attr(llm, "last_output_tokens"),
        "cost": safe_get_attr(llm, "last_cost"),
        "latency_ms": safe_get_attr(llm, "last_latency_ms"),
    }

def normalize_numeric_answer(text):
    if text is None:
        return ""

    s = str(text).strip().lower()
    replacements = {
        "$": "",
        "m": "",
        "meter": "",
        "meters": "",
        "metre": "",
        "metres": "",
        ",": " ",
    }
    for old, new in replacements.items():
        s = s.replace(old, new)

    s = re.sub(r"\s+", " ", s).strip()
    return s

def extract_first_number(text):
    s = normalize_numeric_answer(text)
    if not s:
        return None, s

    m = re.search(r"[-+]?\d+(?:\.\d+)?", s)
    if not m:
        return None, s

    try:
        return float(m.group(0)), s
    except Exception:
        return None, s

def code_verifier(answer_text):
    value, normalized = extract_first_number(answer_text)

    if value is None:
        return False, "hallucination", normalized

    if abs(value - CANONICAL_NUMERIC_ANSWER) <= ABS_TOL:
        return True, None, normalized

    return False, None, normalized

def classify_failure_fp_0004(answer_text):
    value, normalized = extract_first_number(answer_text)

    if value is None:
        return "hallucination"

    # Uploaded task-specific common wrong answers
    if abs(value - 0.234) <= 0.003 or abs(value - 0.233) <= 0.003:
        return "misapplication_of_equation_or_model"

    if abs(value - 0.142) <= 0.003:
        return "failure_to_recognize_key_aspects"

    # Near miss -> likely arithmetic / rounding
    if abs(value - CANONICAL_NUMERIC_ANSWER) <= 0.02:
        return "calculation_error"

    return "misapplication_of_equation_or_model"

# ----------------------------
# Frontier Physics Task 004
# ----------------------------
@kbench.task(
    name="FP-0004 Piecewise Helix Turning Displacement",
    description="Medium classical-mechanics task on a bead constrained to a piecewise-smooth helical wire in a uniformly accelerating non-rotating frame."
)
def fp_0004_piecewise_helix_turning_displacement(llm) -> tuple[int, int]:
    prompt = r"""You are solving a physics problem. Return valid JSON only — no prose outside the JSON.

A rigid wire is mounted to a carriage. In the carriage frame, which translates but does not rotate, the wire lies in 3D and is wrapped around the carriage $x$-axis with constant radius $R$. A bead of mass $m$ slides on the wire without friction and remains constrained to the wire. Gravity is uniform and points in the $-z$ direction: $\mathbf g = -g \hat{\mathbf z}$ with $g=9.80665\ \mathrm{m/s^2}$. The carriage undergoes a constant translational acceleration $\mathbf a = a_x \hat{\mathbf x} + a_y \hat{\mathbf y} + a_z \hat{\mathbf z}$ relative to an inertial lab frame.

The wire is parameterized by $u \in \mathbb{R}$ with
$y(u)=R\cos u$, $z(u)=R\sin u$.

Its $x$-coordinate changes pitch at a known join location $u=u_c$:

Segment 1 (upper helix), for $u \ge u_c$:
$x(u)=b_1 u$

Segment 2 (lower helix with $C^1$ join), for $u \le u_c$:
$x(u)=b_1 u_c + b_2(u-u_c) + (b_1-b_2)L\big(1-e^{(u-u_c)/L}\big)$

At time $t=0$ the bead is clamped at $u_0=-\frac{\pi}{2}$, then released from rest relative to the wire. Define the bead’s horizontal displacement as $\Delta x = x - x_0$, where $x_0=x(u_0)$. Let $\Delta x_{\max}$ mean the first nonzero value of $|\Delta x|$ at which the bead’s speed along the wire becomes zero again after release.

Numerical values:
$R=0.180\ \mathrm{m}$
$u_c=-2.20\ \mathrm{rad}$
$L=0.080\ \mathrm{rad}$
$b_1=0.300\ \mathrm{m/rad}$
$b_2=0.100\ \mathrm{m/rad}$
$a_x=5.40\ \mathrm{m/s^2}$
$a_y=-3.10\ \mathrm{m/s^2}$
$a_z=1.70\ \mathrm{m/s^2}$

Question: What is $\Delta x_{\max}$ in meters, to 3 significant figures?

Return JSON only in the following format:
{
  "final_answer": "<numeric value in meters>"
}"""

    response = llm.prompt(prompt)
    parsed = extract_json(response)

    total_checks = 1
    passed_checks = 0
    final_answer = ""
    normalized_answer = ""
    failure_mode = None

    if parsed is None:
        failure_mode = "hallucination"
    else:
        final_answer = parsed.get("final_answer", "")
        code_result, code_failure, normalized_answer = code_verifier(final_answer)

        if code_result is True:
            passed_checks = 1
        else:
            failure_mode = code_failure or classify_failure_fp_0004(final_answer)

    trace = build_trace(
        task_id="fp_0004",
        llm=llm,
        prompt=prompt,
        response=response,
        parsed=parsed,
        final_answer=final_answer,
        normalized_answer=normalized_answer,
        passed=(passed_checks == 1),
        failure_mode=failure_mode,
    )
    TRACE_LOG.append(trace)

    return (passed_checks, total_checks)

In [ ]:
fp_0004_piecewise_helix_turning_displacement.run(kbench.llm)

In [ ]:
results = fp_0004_piecewise_helix_turning_displacement.evaluate(llm=[kbench.llm])
results.as_dataframe()

In [ ]:
import pandas as pd

trace_df = pd.DataFrame(TRACE_LOG)
trace_df[trace_df["task_id"] == "fp_0004"]